# Notebook 13 -- Variational Algorithms (VQE and QAOA)

Variational quantum algorithms are the **workhorse of the NISQ era**.
Instead of requiring deep circuits with thousands of perfect gates, they
use **shallow parameterized circuits** whose parameters are tuned by a
classical optimizer. This hybrid quantum-classical loop lets us extract
useful results from today's noisy, limited-qubit hardware.

The key insight is the **variational principle**: for any trial state
$|\psi(\theta)\rangle$, the expectation value
$\langle\psi(\theta)|H|\psi(\theta)\rangle \geq E_0$ is an upper bound
on the true ground-state energy $E_0$. By minimizing over $\theta$, we
get the best approximation to the ground state that our circuit can
express.

In this notebook we will:

1. Build **parameterized circuits** with symbolic parameters.
2. Explore **ansatz templates** -- pre-built circuit structures for
   variational algorithms.
3. Run the **Variational Quantum Eigensolver (VQE)** to find the ground
   state energy of a molecular Hamiltonian.
4. Run the **Quantum Approximate Optimization Algorithm (QAOA)** to solve
   a combinatorial optimization problem (MaxCut).
5. Compare **classical optimizers** (Nelder-Mead, SPSA, Adam, L-BFGS).
6. Compute **quantum gradients** via parameter-shift and finite differences.
7. Use **parameter sweeps** to visualize energy landscapes.

### Misconception: Variational algorithms always find the global minimum

Variational algorithms are **not guaranteed** to converge to the true
ground state. The quality of the result depends on (a) the expressibility
of the ansatz -- can it even represent the target state? -- and (b) the
optimization landscape, which may contain local minima and barren plateaus.
Careful ansatz design and optimizer choice are essential.

In [ ]:
import (
	"context"
	"fmt"
	"math"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/algorithm/ansatz"
	"github.com/splch/goqu/algorithm/gradient"
	"github.com/splch/goqu/algorithm/optim"
	"github.com/splch/goqu/algorithm/qaoa"
	"github.com/splch/goqu/algorithm/vqe"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/ir"
	"github.com/splch/goqu/circuit/param"
	"github.com/splch/goqu/sim/pauli"
	"github.com/splch/goqu/sweep"
	"github.com/splch/goqu/viz"
)

## Parameterized Circuits and Symbolic Parameters

A **parameterized circuit** contains gates whose angles are not fixed
numbers but **symbolic parameters** -- named placeholders that can be
bound to concrete values later. This is the foundation of all variational
algorithms: we build a circuit template once, then evaluate it at many
different parameter values during optimization.

In Goqu, `param.New(name)` creates a standalone symbolic parameter, and
`p.Expr()` returns an expression that can be passed to symbolic gate
builders like `b.SymRY(expr, qubit)`. After building the circuit,
`ir.FreeParameters(c)` lists all unbound parameter names, and
`ir.Bind(c, bindings)` substitutes concrete values to produce a
standard circuit that can be simulated.

Let's create a simple 1-qubit parameterized circuit with a single
RY rotation.

In [ ]:
%%
// Create a symbolic parameter
theta := param.New("theta")

// Build a parameterized circuit with a symbolic RY gate
b := builder.New("param-circuit", 1)
b.SymRY(theta.Expr(), 0)
c, err := b.Build()
if err != nil {
	panic(err)
}

// Inspect the free parameters
fmt.Println("Free parameters:", ir.FreeParameters(c))
fmt.Println()
fmt.Println("Parameterized circuit:")
fmt.Println(draw.String(c))

// Bind the parameter to a concrete value and inspect the result
bound, err := ir.Bind(c, map[string]float64{"theta": math.Pi / 4})
if err != nil {
	panic(err)
}

fmt.Println("After binding theta = pi/4:")
fmt.Println(draw.String(bound))
fmt.Println("Free parameters after binding:", ir.FreeParameters(bound))
fmt.Println("\nThe symbolic placeholder has been replaced with a concrete rotation angle.")

## Ansatz Templates

An **ansatz** (German for "approach") is a pre-built parameterized circuit
template designed for variational algorithms. The choice of ansatz
determines:

- **Expressibility**: which states the circuit can represent.
- **Trainability**: whether the optimizer can find good parameters (or
  gets stuck in barren plateaus).
- **Hardware efficiency**: how many gates and how much depth the circuit
  requires.

Goqu provides several standard ansatze:

| Ansatz | Rotations | Params/qubit/rep | Best for |
|:---|:---|:---:|:---|
| **RealAmplitudes** | RY | 1 | Real-valued wavefunctions (chemistry) |
| **EfficientSU2** | RY + RZ | 2 | General states (full SU(2) coverage) |
| **StronglyEntanglingLayers** | Rot(phi,theta,omega) | 3 | Maximum expressibility |
| **BasicEntanglerLayers** | RX | 1 | Simple problems, fast training |

Each ansatz also supports different **entanglement patterns**: `Linear`
(nearest-neighbor CNOTs), `Full` (all-to-all CNOTs), and `Circular`
(linear + wraparound). Let's inspect two of them.

In [ ]:
%%
// RealAmplitudes: RY rotations + CNOT entanglement
ra := ansatz.NewRealAmplitudes(2, 2, ansatz.Linear)
raCircuit, err := ra.Circuit()
if err != nil {
	panic(err)
}
fmt.Println("RealAmplitudes (2 qubits, 2 reps, Linear entanglement):")
fmt.Println(draw.String(raCircuit))
fmt.Printf("Parameters: %d\n", ra.NumParams())
fmt.Printf("Free params: %v\n\n", ir.FreeParameters(raCircuit))

// EfficientSU2: RY + RZ rotations + CNOT entanglement
esu2 := ansatz.NewEfficientSU2(2, 2, ansatz.Full)
esu2Circuit, err := esu2.Circuit()
if err != nil {
	panic(err)
}
fmt.Println("EfficientSU2 (2 qubits, 2 reps, Full entanglement):")
fmt.Println(draw.String(esu2Circuit))
fmt.Printf("Parameters: %d\n", esu2.NumParams())

fmt.Println("\nEfficientSU2 has twice as many parameters as RealAmplitudes")
fmt.Println("because it uses both RY and RZ rotations per qubit per layer.")

## VQE -- Variational Quantum Eigensolver

VQE finds the **ground state energy** of a Hamiltonian $H$ by variationally
minimizing the expectation value $\langle\psi(\theta)|H|\psi(\theta)\rangle$.

The algorithm:

1. **Prepare** a trial state $|\psi(\theta)\rangle$ using a parameterized
   circuit (ansatz).
2. **Measure** the expectation value $\langle H \rangle$ -- the energy.
3. **Optimize** the parameters $\theta$ using a classical optimizer to
   minimize the energy.
4. **Repeat** until convergence.

By the variational principle, the final energy is an **upper bound** on
the true ground state energy $E_0$. The tighter the bound, the better
the ansatz and optimizer performed.

### Building a Hamiltonian: H$_2$ Molecule

The Hamiltonian of a physical system is expressed as a weighted sum of
Pauli strings (tensor products of I, X, Y, Z). For the hydrogen molecule
H$_2$ at its equilibrium bond length (~0.735 angstroms), the electronic
Hamiltonian in the STO-3G basis with Bravyi-Kitaev mapping reduces to
a 2-qubit Pauli sum:

$$H_{\text{H}_2} = c_0 \cdot II + c_1 \cdot ZZ + c_2 \cdot ZI + c_3 \cdot IZ$$

where the coefficients come from the molecular integrals. The exact
ground state energy for this Hamiltonian is approximately $-1.137$ Hartree.

Let's build this Hamiltonian and run VQE.

In [ ]:
%%
// Simplified H2 Hamiltonian at equilibrium bond length
// H = -1.0523732*II + 0.39793742*ZZ + 0.18093119*ZI - 0.18093119*IZ
zz, _ := pauli.Parse("ZZ")
zi, _ := pauli.Parse("ZI")
iz, _ := pauli.Parse("IZ")
ii, _ := pauli.Parse("II")

h2, err := pauli.NewPauliSum([]pauli.PauliString{
	ii.Scale(-1.0523732),
	zz.Scale(0.39793742),
	zi.Scale(0.18093119),
	iz.Scale(-0.18093119),
})
if err != nil {
	panic(err)
}

fmt.Println("H2 Hamiltonian:")
fmt.Println(" ", h2)
fmt.Printf("  Qubits: %d\n\n", h2.NumQubits())

// Run VQE with RealAmplitudes ansatz and Nelder-Mead optimizer
ctx := context.Background()
ra := ansatz.NewRealAmplitudes(2, 1, ansatz.Linear)

result, err := vqe.Run(ctx, vqe.Config{
	Hamiltonian: h2,
	Ansatz:      ra,
	Optimizer:   &optim.NelderMead{},
})
if err != nil {
	panic(err)
}

fmt.Printf("Ground state energy: %.6f Hartree\n", result.Energy)
fmt.Printf("Converged: %v\n", result.Converged)
fmt.Printf("Iterations: %d\n", result.NumIters)
fmt.Printf("Function evaluations: %d\n", result.NumEvals)
fmt.Printf("Optimal parameters: %v\n", result.OptimalParams)
fmt.Printf("\nExact ground state energy ~ -1.137 Hartree\n")
fmt.Printf("VQE error: %.6f Hartree\n", result.Energy - (-1.137))

In [ ]:
%%
// Visualize the VQE convergence history
fmt.Println("VQE Convergence History")
fmt.Println("=======================")
fmt.Printf("%6s  %12s\n", "Eval", "Energy")
fmt.Println("--------------------")

// Show every 10th evaluation to keep output manageable
for i, e := range result.History {
	if i%(len(result.History)/10+1) == 0 || i == len(result.History)-1 {
		fmt.Printf("%6d  %12.6f\n", i+1, e)
	}
}

fmt.Printf("\nTotal evaluations: %d\n", len(result.History))
fmt.Printf("Final energy: %.6f\n", result.History[len(result.History)-1])
fmt.Println("\nThe energy decreases monotonically toward the ground state.")
fmt.Println("The optimizer explores the parameter landscape, always seeking lower energy.")

## QAOA -- Quantum Approximate Optimization Algorithm

While VQE targets physics problems (finding ground states), **QAOA**
targets **combinatorial optimization** problems -- finding the minimum
of a cost function over binary strings.

QAOA builds a circuit with $p$ alternating layers:

1. **Cost layer**: $e^{-i \gamma_k C}$ -- encodes the problem.
2. **Mixer layer**: $e^{-i \beta_k B}$ -- explores the solution space.

The circuit starts in $|+\rangle^{\otimes n}$ (uniform superposition over
all bitstrings) and the angles $(\gamma_1, \ldots, \gamma_p,
\beta_1, \ldots, \beta_p)$ are optimized classically. As $p \to \infty$,
QAOA is guaranteed to converge to the optimal solution (by the quantum
adiabatic theorem), but in practice we use small $p$ on NISQ devices.

### MaxCut Problem

A classic application of QAOA is **MaxCut**: given a graph, partition
the vertices into two sets to maximize the number of edges crossing the
partition. The cost Hamiltonian is:

$$C = \sum_{(i,j) \in E} \frac{1}{2}(I - Z_i Z_j)$$

Each edge contributes $+1$ when vertices $i$ and $j$ are in different
partitions and $0$ when they are in the same partition.

In [ ]:
%%
// QAOA for MaxCut on a triangle graph: edges (0,1), (1,2), (0,2)
edges := [][2]int{{0, 1}, {1, 2}, {0, 2}}
costH, err := qaoa.MaxCutHamiltonian(edges, 3)
if err != nil {
	panic(err)
}

fmt.Println("MaxCut Hamiltonian (triangle graph):")
fmt.Println(" ", costH)
fmt.Printf("  Qubits: %d\n\n", costH.NumQubits())

ctx := context.Background()
qaoaResult, err := qaoa.Run(ctx, qaoa.Config{
	CostHamiltonian: costH,
	Layers:          2,
	Optimizer:       &optim.NelderMead{},
	Shots:           1000,
})
if err != nil {
	panic(err)
}

fmt.Printf("Best bitstring: %s\n", qaoaResult.BestBitstring)
fmt.Printf("Best cost: %.2f\n", qaoaResult.BestCost)
fmt.Printf("Optimal value: %.4f\n", qaoaResult.OptimalValue)
fmt.Printf("Converged: %v, Iterations: %d\n", qaoaResult.Converged, qaoaResult.NumIters)

fmt.Println("\nFor a triangle (K3), MaxCut = 2 (any partition cutting 2 of 3 edges).")
fmt.Println("Valid maxcut bitstrings: 001, 010, 100, 110, 101, 011.")
fmt.Println("QAOA should find one of these.")

## Optimizers

The choice of classical optimizer significantly affects VQE and QAOA
performance. Goqu provides four optimizers:

| Optimizer | Type | Evals/iter | Best for |
|:---|:---|:---:|:---|
| **Nelder-Mead** | Gradient-free (simplex) | $n+1$ | Small parameter spaces, no gradient available |
| **SPSA** | Gradient-free (stochastic) | 2 | Noisy cost functions, large parameter spaces |
| **Adam** | Gradient-based | 1 + gradient | Smooth landscapes, many parameters |
| **L-BFGS** | Gradient-based (quasi-Newton) | 1 + gradient | Smooth landscapes, fast convergence |

Gradient-free methods (Nelder-Mead, SPSA) only need the objective value.
Gradient-based methods (Adam, L-BFGS) also need the gradient
$\nabla_\theta \langle H \rangle$, which can be computed via parameter-shift
or finite differences. Let's compare all four on the same H$_2$ VQE problem.

In [ ]:
%%
// Compare optimizers on the H2 VQE problem
ctx := context.Background()

// Rebuild the H2 Hamiltonian
zz, _ := pauli.Parse("ZZ")
zi, _ := pauli.Parse("ZI")
iz, _ := pauli.Parse("IZ")
ii, _ := pauli.Parse("II")

h2, _ := pauli.NewPauliSum([]pauli.PauliString{
	ii.Scale(-1.0523732),
	zz.Scale(0.39793742),
	zi.Scale(0.18093119),
	iz.Scale(-0.18093119),
})

// Build gradient function for gradient-based optimizers
raOpt := ansatz.NewRealAmplitudes(2, 1, ansatz.Linear)
raOptCircuit, _ := raOpt.Circuit()
paramNames := ir.FreeParameters(raOptCircuit)
costFn := gradient.CostFunc(raOptCircuit, h2, paramNames)
psGrad := gradient.ParameterShift(costFn)

type optimEntry struct {
	name string
	opt  optim.Optimizer
	grad optim.GradientFunc
}

optimizers := []optimEntry{
	{"Nelder-Mead", &optim.NelderMead{}, nil},
	{"SPSA", &optim.SPSA{}, nil},
	{"Adam", &optim.Adam{LR: 0.1}, psGrad},
	{"L-BFGS", &optim.LBFGS{}, psGrad},
}

fmt.Println("Optimizer Comparison on H2 VQE")
fmt.Println("==============================")
fmt.Printf("%-14s  %12s  %6s  %8s  %10s\n", "Optimizer", "Energy", "Iters", "Converged", "Error")
fmt.Println("--------------------------------------------------------------")

for _, entry := range optimizers {
	ra := ansatz.NewRealAmplitudes(2, 1, ansatz.Linear)
	res, err := vqe.Run(ctx, vqe.Config{
		Hamiltonian: h2,
		Ansatz:      ra,
		Optimizer:   entry.opt,
		Gradient:    entry.grad,
	})
	if err != nil {
		fmt.Printf("%-14s  error: %v\n", entry.name, err)
		continue
	}
	fmt.Printf("%-14s  %12.6f  %6d  %8v  %10.6f\n",
		entry.name, res.Energy, res.NumIters, res.Converged, res.Energy-(-1.137))
}

fmt.Println("\nGradient-based methods (Adam, L-BFGS) typically converge faster")
fmt.Println("but require computing the gradient at each step.")

## Gradient Methods

Gradient-based optimizers need $\nabla_\theta \langle H \rangle$.
On a quantum computer, we cannot directly differentiate a quantum circuit,
but we have two strategies:

### Parameter-Shift Rule

For gates of the form $e^{-i\theta G/2}$ where $G$ has eigenvalues
$\pm 1$ (which includes RX, RY, RZ), the **exact** gradient is:

$$\frac{\partial}{\partial \theta_j} \langle H \rangle =
\frac{1}{2} \left[ f(\theta_j + \pi/2) - f(\theta_j - \pi/2) \right]$$

This requires **2N function evaluations** per gradient (N = number of
parameters), but gives the **exact analytical gradient**.

### Finite Differences

A simpler but approximate method:

$$\frac{\partial}{\partial \theta_j} \langle H \rangle \approx
\frac{f(\theta_j + h) - f(\theta_j - h)}{2h}$$

This also needs 2N evaluations but introduces a bias controlled by $h$.
Smaller $h$ reduces bias but increases sensitivity to shot noise.

In [ ]:
%%
// Compare parameter-shift and finite-difference gradients
zz, _ := pauli.Parse("ZZ")
zi, _ := pauli.Parse("ZI")
iz, _ := pauli.Parse("IZ")
ii, _ := pauli.Parse("II")

h2, _ := pauli.NewPauliSum([]pauli.PauliString{
	ii.Scale(-1.0523732),
	zz.Scale(0.39793742),
	zi.Scale(0.18093119),
	iz.Scale(-0.18093119),
})

raGrad := ansatz.NewRealAmplitudes(2, 1, ansatz.Linear)
raGradCircuit, _ := raGrad.Circuit()
paramNames := ir.FreeParameters(raGradCircuit)
costFn := gradient.CostFunc(raGradCircuit, h2, paramNames)

// Create both gradient methods
psGrad := gradient.ParameterShift(costFn)
fdGrad := gradient.FiniteDifference(costFn, 1e-5)

// Evaluate at a test point
testPoint := make([]float64, len(paramNames))
for i := range testPoint {
	testPoint[i] = 0.5 * float64(i+1)
}

psResult := psGrad(testPoint)
fdResult := fdGrad(testPoint)

fmt.Println("Gradient Comparison at test point")
fmt.Println("=================================")
fmt.Printf("Cost at test point: %.6f\n\n", costFn(testPoint))

fmt.Printf("%-12s  %14s  %14s  %14s\n", "Parameter", "Param-Shift", "Finite-Diff", "Difference")
fmt.Println("----------------------------------------------------------")

for i, name := range paramNames {
	diff := math.Abs(psResult[i] - fdResult[i])
	fmt.Printf("%-12s  %14.8f  %14.8f  %14.2e\n", name, psResult[i], fdResult[i], diff)
}

fmt.Println("\nBoth methods agree to high precision.")
fmt.Println("Parameter-shift gives the exact gradient; finite differences are approximate.")
fmt.Println("On real hardware with shot noise, parameter-shift is generally preferred.")

## Parameter Sweeps

To understand how the energy depends on a single parameter, we can
**sweep** it across a range of values while holding everything else
fixed. This creates an **energy landscape** -- a 1D slice through the
high-dimensional parameter space.

Goqu's `sweep.Linspace` generates evenly-spaced values for a named
parameter, and `sweep.RunSim` evaluates the parameterized circuit at
each point using statevector simulation.

Energy landscapes reveal important features:
- **Global minima** where the optimizer should converge.
- **Local minima** that can trap gradient-based methods.
- **Periodicity** from the underlying rotation gates.

In [ ]:
%%
// Sweep a single parameter of a 1-qubit parameterized circuit
// to visualize the energy landscape
theta := param.New("theta")
b := builder.New("sweep-demo", 1)
b.SymRY(theta.Expr(), 0)
b.MeasureAll()
sweepCircuit, _ := b.Build()

ctx := context.Background()
sw := sweep.Linspace{Key: "theta", Start: 0, Stop: 2 * math.Pi, Count: 50}
results, err := sweep.RunSim(ctx, sweepCircuit, 1000, sw)
if err != nil {
	panic(err)
}

fmt.Println("Parameter Sweep: RY(theta) on |0>")
fmt.Println("===================================")
fmt.Printf("%8s  %8s  %8s\n", "theta", "P(|0>)", "P(|1>)")
fmt.Println("--------------------------")

for i, res := range results {
	if res.Err != nil {
		continue
	}
	if i%5 == 0 || i == len(results)-1 {
		total := 0
		for _, c := range res.Counts {
			total += c
		}
		p0 := float64(res.Counts["0"]) / float64(total)
		p1 := float64(res.Counts["1"]) / float64(total)
		fmt.Printf("%8.3f  %8.3f  %8.3f\n", res.Bindings["theta"], p0, p1)
	}
}

fmt.Println("\nRY(theta)|0> = cos(theta/2)|0> + sin(theta/2)|1>")
fmt.Println("At theta=0: pure |0>. At theta=pi: pure |1>. At theta=2pi: back to |0>.")

## Predict, Then Verify

**Question:** What happens if you use too few QAOA layers for MaxCut on
a triangle graph?

**Prediction:** A triangle (K3) has MaxCut = 2. With $p = 1$ layer, QAOA
has only 2 parameters ($\gamma_1, \beta_1$) to work with, which limits
the quality of the approximation. The **approximation ratio**
$r = C_{\text{QAOA}} / C_{\text{max}}$ should improve as we increase $p$.
For $p = 1$, the ratio is bounded away from 1; for larger $p$, it should
approach 1.

Let's sweep $p$ from 1 to 5 and track the approximation ratio.

In [ ]:
%%
// QAOA with increasing layers on a triangle graph
edges := [][2]int{{0, 1}, {1, 2}, {0, 2}}
costH, _ := qaoa.MaxCutHamiltonian(edges, 3)
maxCut := 2.0 // Known optimal for K3

ctx := context.Background()

fmt.Println("QAOA Layer Sweep on Triangle Graph (MaxCut = 2)")
fmt.Println("================================================")
fmt.Printf("%8s  %12s  %12s  %14s  %10s\n",
	"Layers", "Best Cost", "Bitstring", "Approx Ratio", "Converged")
fmt.Println("--------------------------------------------------------------")

for p := 1; p <= 5; p++ {
	res, err := qaoa.Run(ctx, qaoa.Config{
		CostHamiltonian: costH,
		Layers:          p,
		Optimizer:       &optim.NelderMead{},
		Shots:           2000,
	})
	if err != nil {
		fmt.Printf("%8d  error: %v\n", p, err)
		continue
	}
	// MaxCut cost is negated in the Hamiltonian, so negate BestCost
	cutValue := -res.BestCost
	ratio := cutValue / maxCut
	fmt.Printf("%8d  %12.4f  %12s  %14.4f  %10v\n",
		p, res.BestCost, res.BestBitstring, ratio, res.Converged)
}

fmt.Println("\nMore layers give the optimizer more degrees of freedom.")
fmt.Println("The approximation ratio should approach 1.0 as p increases.")
fmt.Println("However, more layers also mean deeper circuits and harder optimization.")

---

## Exercises

### Exercise 1 -- Compare Ansatze on the H$_2$ Hamiltonian

Run VQE on the H$_2$ Hamiltonian using three different ansatze
(RealAmplitudes, EfficientSU2, StronglyEntanglingLayers) and compare
the resulting ground state energies. Which ansatz gets closest to the
exact value, and at what computational cost (parameters, iterations)?

In [ ]:
%%
// Exercise 1: Compare ansatze on H2
ctx := context.Background()

zz, _ := pauli.Parse("ZZ")
zi, _ := pauli.Parse("ZI")
iz, _ := pauli.Parse("IZ")
ii, _ := pauli.Parse("II")

h2, _ := pauli.NewPauliSum([]pauli.PauliString{
	ii.Scale(-1.0523732),
	zz.Scale(0.39793742),
	zi.Scale(0.18093119),
	iz.Scale(-0.18093119),
})

type ansatzEntry struct {
	name   string
	ansatz ansatz.Ansatz
}

ansatze := []ansatzEntry{
	{"RealAmplitudes", ansatz.NewRealAmplitudes(2, 2, ansatz.Linear)},
	{"EfficientSU2", ansatz.NewEfficientSU2(2, 2, ansatz.Full)},
	{"StronglyEntangling", ansatz.NewStronglyEntanglingLayers(2, 2)},
}

fmt.Println("Exercise 1: Ansatz Comparison on H2")
fmt.Println("====================================")
fmt.Printf("%-22s  %6s  %12s  %6s  %10s\n",
	"Ansatz", "Params", "Energy", "Iters", "Error")
fmt.Println("--------------------------------------------------------------")

for _, entry := range ansatze {
	res, err := vqe.Run(ctx, vqe.Config{
		Hamiltonian: h2,
		Ansatz:      entry.ansatz,
		Optimizer:   &optim.NelderMead{},
	})
	if err != nil {
		fmt.Printf("%-22s  error: %v\n", entry.name, err)
		continue
	}
	fmt.Printf("%-22s  %6d  %12.6f  %6d  %10.6f\n",
		entry.name, entry.ansatz.NumParams(), res.Energy,
		res.NumIters, res.Energy-(-1.137))
}

fmt.Println("\nMore expressive ansatze can represent the ground state more accurately,")
fmt.Println("but more parameters make optimization harder and slower.")

### Exercise 2 -- QAOA MaxCut on a 4-Node Graph

Run QAOA on a 4-node graph (a square with one diagonal):
edges (0,1), (1,2), (2,3), (3,0), (0,2). The maximum cut for this graph
is 4 (partition {0,2} vs {1,3}). Try different numbers of QAOA layers
and see how the approximation quality improves.

In [ ]:
%%
// Exercise 2: QAOA MaxCut on a 4-node graph (square + diagonal)
edges := [][2]int{{0, 1}, {1, 2}, {2, 3}, {3, 0}, {0, 2}}
costH, err := qaoa.MaxCutHamiltonian(edges, 4)
if err != nil {
	panic(err)
}
maxCut := 4.0 // Optimal: partition {0,2} vs {1,3} cuts all 5 edges? No -- (0,2) is same side, so 4 edges.

ctx := context.Background()

fmt.Println("Exercise 2: QAOA MaxCut on 4-Node Graph")
fmt.Println("========================================")
fmt.Println("Graph: square with diagonal (0,2)")
fmt.Println("Edges: (0,1), (1,2), (2,3), (3,0), (0,2)")
fmt.Printf("Optimal MaxCut: %.0f\n\n", maxCut)

fmt.Printf("%8s  %12s  %12s  %14s  %10s\n",
	"Layers", "Best Cost", "Bitstring", "Approx Ratio", "Converged")
fmt.Println("--------------------------------------------------------------")

for p := 1; p <= 4; p++ {
	res, err := qaoa.Run(ctx, qaoa.Config{
		CostHamiltonian: costH,
		Layers:          p,
		Optimizer:       &optim.NelderMead{},
		Shots:           2000,
	})
	if err != nil {
		fmt.Printf("%8d  error: %v\n", p, err)
		continue
	}
	cutValue := -res.BestCost
	ratio := cutValue / maxCut
	fmt.Printf("%8d  %12.4f  %12s  %14.4f  %10v\n",
		p, res.BestCost, res.BestBitstring, ratio, res.Converged)
}

fmt.Println("\nWith 4 nodes and 5 edges, this problem is harder than the triangle.")
fmt.Println("More QAOA layers improve the approximation quality.")

## Key Takeaways

1. **Variational algorithms** use shallow parameterized circuits optimized
   by classical computers. They are the leading approach for near-term
   quantum hardware because they tolerate limited qubit counts and gate
   fidelities.

2. **Parameterized circuits** use symbolic parameters (`param.New`) that
   are bound to concrete values via `ir.Bind`. This separation of
   structure and values is the foundation of all variational methods.

3. **Ansatz design** is critical. RealAmplitudes, EfficientSU2, and
   StronglyEntanglingLayers offer increasing expressibility at the cost
   of more parameters. The right choice depends on the problem.

4. **VQE** finds ground state energies by minimizing
   $\langle\psi(\theta)|H|\psi(\theta)\rangle$. The variational principle
   guarantees the result is an upper bound on $E_0$.

5. **QAOA** solves combinatorial optimization problems by alternating
   cost and mixer layers. More layers ($p$) improve the approximation
   but increase circuit depth.

6. **Optimizer choice matters**. Nelder-Mead and SPSA are gradient-free;
   Adam and L-BFGS use gradients for faster convergence. The
   parameter-shift rule gives exact quantum gradients.

7. **Energy landscapes** can have local minima, barren plateaus, and
   symmetries that make optimization challenging. Understanding the
   landscape (via parameter sweeps) helps choose initial parameters
   and optimizer settings.

8. **There are no guarantees**: variational algorithms may converge to
   local minima or barren plateaus rather than the global optimum.
   Multiple random initializations and careful ansatz design mitigate
   this, but do not eliminate it.

---

**Next up:** Notebook 14, where we will explore error mitigation
techniques for improving the accuracy of noisy quantum computations.